# 02 Finite-Sample Inference — ALARM
The finite-sample experiment uses two fixed data streams: $D_J\sim P(G)$ for J-Method, and $D_C\sim P(R\mid C)$ shared by the four FGCR-C methods and IC. All local CPTs are estimated using BDeu posterior means with the equivalent sample size fixed at `ISS=10`.

## 1. Load the Shared Core from Notebook 00


In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
COMMON_CORE_PATH = PROJECT_ROOT / "00_FGCR_Common_Core.ipynb"
if not COMMON_CORE_PATH.exists():
    raise FileNotFoundError("Missing 00_FGCR_Common_Core.ipynb in the project root")

_common_notebook = json.loads(COMMON_CORE_PATH.read_text(encoding="utf-8"))
_loaded_common_cells = 0
for _cell in _common_notebook["cells"]:
    _tags = _cell.get("metadata", {}).get("tags", [])
    if _cell.get("cell_type") == "code" and "common_core" in _tags:
        exec(compile("".join(_cell.get("source", [])), str(COMMON_CORE_PATH), "exec"), globals())
        _loaded_common_cells += 1

if _loaded_common_cells != 7:
    raise RuntimeError(f"Expected 7 common_core cells, loaded {_loaded_common_cells}")


## 2. Finite-Sample Computation Core


In [2]:
#RUN_MODE = os.environ.get("FGCR_RUN_MODE", "quick").strip().lower()
RUN_MODE = os.environ.get("FGCR_RUN_MODE", "formal").strip().lower()
RESUME_CHECKPOINTS = os.environ.get("FGCR_RESUME", "1") != "0"
KEEP_CHECKPOINTS = os.environ.get("FGCR_KEEP_CHECKPOINTS", "1") != "0"
SAMPLER_EXACT_TOLERANCE = 1e-10

# Forward-sample a multi-state BN according to the graph's true topological order; the returned column order always matches graph.nodes.
def sample_bn(
    graph: DAG,
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
    n: int,
    seed: int,
) -> tuple[np.ndarray, tuple[Node, ...]]:
    if n < 1:
        raise ValueError("Sample size must be positive")
    validate_cpts(graph, cpts, parent_order)
    rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence(seed)))

    # Sample columns use the stable graph.nodes order, while the sampling loop follows the actual topological order.
    columns = tuple(graph.nodes)
    column_index = {node: index for index, node in enumerate(columns)}
    sample = np.empty((n, len(columns)), dtype=np.int16)
    for node in topological_sort(graph):
        parents = tuple(parent_order[node])
        parent_code = np.zeros(n, dtype=np.int64)
        for parent in parents:
            # The mixed-radix encoding exactly matches the C-order parent-configuration ordering used when reshaping the CPT.
            parent_code = (
                parent_code * CARDINALITY[parent]
                + sample[:, column_index[parent]]
            )
        rows = np.asarray(cpts[node]).reshape(CARDINALITY[node], -1).T
        probabilities = rows[parent_code]
        uniforms = rng.random(n)
        cumulative = np.cumsum(probabilities, axis=1)
        draws = (uniforms[:, None] > cumulative).sum(axis=1)
        sample[:, column_index[node]] = np.minimum(
            draws, CARDINALITY[node] - 1
        )
    return sample, columns

# Bind the sample column definition and raw integer matrix into a single hash, proving that the four FGCR methods and IC share exactly the same D_C.
# Construct an audit hash that binds the sample columns to the integer matrix.
def sample_sha256(sample: np.ndarray, columns: Sequence[Node]) -> str:
    digest = hashlib.sha256()
    digest.update("|".join(columns).encode("utf-8"))
    digest.update(np.ascontiguousarray(sample, dtype=np.int16).tobytes())
    return digest.hexdigest()

# Before actually sampling D_C, verify that the order_1 conditional BN exactly represents P(TARGET|C) for the target joint distribution.
# For each parameter set, verify that the order_1 conditional-sampling BN agrees with the independent ground truth on the target joint distribution.
def validate_conditional_sampler(
    sampler_cpts: Mapping[Node, np.ndarray],
    sampler_parent_order: Mapping[Node, Sequence[Node]],
    truth: np.ndarray,
    parameter_id: int,
    tolerance: float = SAMPLER_EXACT_TOLERANCE,
) -> dict[str, float]:
    # sampler_cpts already compiles the condition C into the local mechanisms, so evidence must not be passed again during the query.
    sampler_target = ve_query(
        sampler_cpts,
        sampler_parent_order,
        TARGET,
    )
    sampler_errors = distribution_errors(truth, sampler_target)
    if max(sampler_errors.values()) > tolerance:
        raise AssertionError(
            "Conditional sampler mismatch for parameter "
            f"{parameter_id}: {sampler_errors}"
        )
    return sampler_errors

# Estimate BDeu posterior-mean CPTs on an arbitrary multi-state DAG.
# Implement generic BDeu posterior means using node cardinalities and the number of parent configurations.
def fit_bdeu(
    sample: np.ndarray,
    sample_columns: Sequence[Node],
    graph: DAG,
    parent_order: Mapping[Node, Sequence[Node]],
    iss: float = ISS,
) -> CPTMap:
    sample = np.asarray(sample, dtype=np.int16)
    if sample.ndim != 2 or sample.shape[1] != len(sample_columns):
        raise ValueError("Sample shape does not match sample_columns")
    if not set(graph.nodes).issubset(set(sample_columns)):
        raise ValueError("Sample omits one or more graph nodes")
    if iss <= 0:
        raise ValueError("BDeu equivalent sample size must be positive")

    column_index = {node: index for index, node in enumerate(sample_columns)}
    fitted: CPTMap = {}
    for node in graph.nodes:
        parents = tuple(parent_order[node])
        response_card = CARDINALITY[node]
        parent_cards = [CARDINALITY[parent] for parent in parents]
        n_parent_configs = math.prod(parent_cards)

         # Encode each row's parent states as an integer in 0,...,q_v-1, then combine it with the node state into a single counting index.
        parent_code = np.zeros(sample.shape[0], dtype=np.int64)
        for parent in parents:
            parent_code = (
                parent_code * CARDINALITY[parent]
                + sample[:, column_index[parent]]
            )
        node_state = sample[:, column_index[node]].astype(np.int64)
        flat_index = parent_code * response_card + node_state
        counts = np.bincount(
            flat_index,
            minlength=n_parent_configs * response_card,
        ).reshape(n_parent_configs, response_card)

         # BDeu distributes ISS uniformly across the r_v q_v cells and returns the posterior mean.
        alpha_cell = iss / (response_card * n_parent_configs)
        posterior = (counts + alpha_cell) / (
            counts.sum(axis=1, keepdims=True) + iss / n_parent_configs
        )
        fitted[node] = posterior.reshape(
            (*parent_cards, response_card)
        ).transpose((-1, *range(len(parent_cards))))
    validate_cpts(graph, fitted, parent_order)
    return fitted

# Freeze the six method results that must appear for every parameter × n × replicate instance.
# Specify the J, four-order FGCR-C, and IC results required for each repetition.
def finite_sample_methods() -> tuple[tuple[str, str], ...]:
    return (
        ("J", "none"),
        ("FGCR-C", "order_1"),
        ("FGCR-C", "order_2"),
        ("FGCR-C", "order_3"),
        ("FGCR-C", "order_4"),
        ("IC", "none"),
    )

# Standardize one estimate into a runs-sheet row, with error consistently defined as estimate - truth.
# Standardize finite-sample repetition-level fields and error definitions.
def finite_sample_row(
    *,
    run_mode: str,
    parameter_id: int,
    sample_size: int,
    replicate: int,
    method: str,
    order_id: str,
    graph: DAG,
    sample_source: str,
    sample_hash: str,
    joint_seed: int,
    conditional_seed: int,
    p_true: float,
    p_estimate: float,
    parameter_sha256: str,
    parameter_bank_sha256: str,
    protocol_sha256: str,
) -> dict[str, object]:
    error = p_estimate - p_true
    return {
        "experiment_id": 2,
        "run_mode": run_mode,
        "parameter_id": parameter_id,
        "sample_size": sample_size,
        "replicate": replicate,
        "method": method,
        "order_id": order_id,
        "graph_source": (
            "original_G" if method == "J"
            else "FGCR_generated_H_R" if method == "FGCR-C"
            else "original_induced_G_R"
        ),
        "edge_count": len(graph.edges),
        "parameter_dimension": parameter_dimension(graph),
        "structure_sha256": edge_sha256(graph),
        "sample_source": sample_source,
        "sample_sha256": sample_hash,
        "joint_seed": joint_seed,
        "conditional_seed": conditional_seed,
        "iss": ISS,
        "p_true": p_true,
        "p_estimate": p_estimate,
        "error": error,
        "squared_error": error * error,
        "absolute_error": abs(error),
        "parameter_sha256": parameter_sha256,
        "parameter_bank_sha256": parameter_bank_sha256,
        "protocol_sha256": protocol_sha256,
    }

# Run all sample sizes, repetitions, and six methods for one parameter set.
# Complete the ground truth, two data streams, and all repeated estimates for a single parameter set.
def run_experiment2_parameter(
    *,
    parameter_id: int,
    source_cpts: Mapping[Node, np.ndarray],
    settings: Mapping[str, object],
    parameter_bank_sha256: str,
    protocol_sha256: str,
    run_mode: str,
) -> pd.DataFrame:
    parameter_hash = cpts_sha256(source_cpts, PARENT_ORDER)
    truth = ancestral_enumeration_query(
        source_cpts, ALARM_DAG, PARENT_ORDER, TARGET, CONDITION
    )
    truth_ve = ve_query(source_cpts, PARENT_ORDER, TARGET, CONDITION)
    if max(distribution_errors(truth, truth_ve).values()) > 1e-10:
        raise AssertionError(f"Truth oracle mismatch for parameter {parameter_id}")
    p_true = event_probability(truth, TARGET, TARGET_EVENT)

     # The exact order_1 H_R is used only as a forward sampler for P(R|C); all methods still use the same D_C.
    sampler_graph = OBSERVATIONAL_STRUCTURES["order_1"].H_R
    sampler_cpts, sampler_parent_order = compile_conditional_cpts(
        source_cpts, PARENT_ORDER, sampler_graph, CONDITION
    )

    # This assertion is performed independently for every parameter_id actually evaluated; any sampler drift stops execution before sampling.
    validate_conditional_sampler(
        sampler_cpts,
        sampler_parent_order,
        truth,
        parameter_id,
    )
    rows: list[dict[str, object]] = []

    for sample_size in settings["sample_sizes"]:
        for replicate in range(1, int(settings["repetitions"]) + 1):
            joint_seed = derive_seed(2, parameter_id, sample_size, replicate, 1)
            conditional_seed = derive_seed(2, parameter_id, sample_size, replicate, 2)

            # D_J is sampled from the original joint BN, while D_C is sampled from the exact conditional BN; the two datasets use independent random streams.
            data_j, columns_j = sample_bn(
                ALARM_DAG, source_cpts, PARENT_ORDER, sample_size, joint_seed
            )
            data_c, columns_c = sample_bn(
                sampler_graph,
                sampler_cpts,
                sampler_parent_order,
                sample_size,
                conditional_seed,
            )
            hash_j = sample_sha256(data_j, columns_j)
            hash_c = sample_sha256(data_c, columns_c)

            # J：fit the joint CPTs on G and then compute P(TARGET|CONDITION).
            fitted_j = fit_bdeu(data_j, columns_j, ALARM_DAG, PARENT_ORDER, ISS)
            estimate_j = ve_query(fitted_j, PARENT_ORDER, TARGET, CONDITION)
            p_j = event_probability(estimate_j, TARGET, TARGET_EVENT)
            rows.append(finite_sample_row(
                run_mode=run_mode,
                parameter_id=parameter_id,
                sample_size=sample_size,
                replicate=replicate,
                method="J",
                order_id="none",
                graph=ALARM_DAG,
                sample_source="joint_sample_D_J",
                sample_hash=hash_j,
                joint_seed=joint_seed,
                conditional_seed=conditional_seed,
                p_true=p_true,
                p_estimate=p_j,
                parameter_sha256=parameter_hash,
                parameter_bank_sha256=parameter_bank_sha256,
                protocol_sha256=protocol_sha256,
            ))

            # Fit the local CPTs of each of the four H_R graphs separately using exactly the same D_C.
            for order_id, result in OBSERVATIONAL_STRUCTURES.items():
                hr_parent_order = parent_order_from_graph(result.H_R)
                fitted_hr = fit_bdeu(
                    data_c, columns_c, result.H_R, hr_parent_order, ISS
                )
                estimate_hr = ve_query(fitted_hr, hr_parent_order, TARGET)
                p_hr = event_probability(estimate_hr, TARGET, TARGET_EVENT)
                rows.append(finite_sample_row(
                    run_mode=run_mode,
                    parameter_id=parameter_id,
                    sample_size=sample_size,
                    replicate=replicate,
                    method="FGCR-C",
                    order_id=order_id,
                    graph=result.H_R,
                    sample_source="shared_conditional_sample_D_C",
                    sample_hash=hash_c,
                    joint_seed=joint_seed,
                    conditional_seed=conditional_seed,
                    p_true=p_true,
                    p_estimate=p_hr,
                    parameter_sha256=parameter_hash,
                    parameter_bank_sha256=parameter_bank_sha256,
                    protocol_sha256=protocol_sha256,
                ))

            # IC uses the same D_C but restricts the structure to G_R, allowing the large-sample error plateau to be observed.
            ic_parent_order = parent_order_from_graph(G_R)
            fitted_ic = fit_bdeu(data_c, columns_c, G_R, ic_parent_order, ISS)
            estimate_ic = ve_query(fitted_ic, ic_parent_order, TARGET)
            p_ic = event_probability(estimate_ic, TARGET, TARGET_EVENT)
            rows.append(finite_sample_row(
                run_mode=run_mode,
                parameter_id=parameter_id,
                sample_size=sample_size,
                replicate=replicate,
                method="IC",
                order_id="none",
                graph=G_R,
                sample_source="shared_conditional_sample_D_C",
                sample_hash=hash_c,
                joint_seed=joint_seed,
                conditional_seed=conditional_seed,
                p_true=p_true,
                p_estimate=p_ic,
                parameter_sha256=parameter_hash,
                parameter_bank_sha256=parameter_bank_sha256,
                protocol_sha256=protocol_sha256,
            ))

    frame = pd.DataFrame(rows)
    # For every experimental instance, the four FGCR methods and IC must have exactly one unique D_C hash.
    shared_counts = (
        frame.loc[frame["sample_source"] == "shared_conditional_sample_D_C"]
        .groupby(["parameter_id", "sample_size", "replicate"])["sample_sha256"]
        .nunique()
    )
    if not shared_counts.eq(1).all():
        raise AssertionError("FGCR-C and IC did not share exactly one D_C")
    return frame

# Round floating-point columns to 12 decimal places before hashing, so checkpoint identity is unaffected by Excel decimal round trips.
# Generate checkpoint content identity that tolerates Excel decimal round trips.
def checkpoint_content_sha256(frame: pd.DataFrame) -> str:
    stable = frame.copy()
    for column in stable.select_dtypes(include=["number"]).columns:
        stable[column] = stable[column].round(12)
    return dataframe_sha256(
        stable,
        sort_by=["parameter_id", "sample_size", "replicate", "method", "order_id"],
    )

# Bind checkpoint identity to the parameter set, run scale, parameter bank, and complete protocol.
# Bind the checkpoint to its parameter set, experiment scale, parameter bank, and protocol.
def checkpoint_identity(
    parameter_id: int,
    settings: Mapping[str, object],
    parameter_sha256: str,
    parameter_bank_sha256: str,
    protocol_sha256: str,
) -> dict[str, object]:
    return {
        "checkpoint_schema_version": 1,
        "experiment_id": 2,
        "parameter_id": parameter_id,
        "sample_sizes_json": json.dumps(settings["sample_sizes"]),
        "repetitions": int(settings["repetitions"]),
        "expected_rows": len(settings["sample_sizes"]) * int(settings["repetitions"]) * 6,
        "parameter_sha256": parameter_sha256,
        "parameter_bank_sha256": parameter_bank_sha256,
        "protocol_sha256": protocol_sha256,
    }

# Atomically write a single-parameter checkpoint and store the logical content hash in metadata.
# Atomically save checkpoint results for one parameter set.
def write_checkpoint(
    path: Path,
    frame: pd.DataFrame,
    identity: Mapping[str, object],
) -> None:
    metadata = pd.DataFrame([{
        **dict(identity),
        "content_sha256": checkpoint_content_sha256(frame),
    }])
    write_excel_atomic(path, {"runs": frame, "metadata": metadata})

# Restore a checkpoint only when its identity, row count, method combinations, and content hash are all valid.
# Restore only checkpoints whose identity and content match completely.
def read_valid_checkpoint(
    path: Path,
    identity: Mapping[str, object],
    settings: Mapping[str, object],
) -> pd.DataFrame | None:
    if not path.exists():
        return None
    try:
        metadata = pd.read_excel(path, sheet_name="metadata")
        if len(metadata) != 1:
            return None
        record = metadata.iloc[0].to_dict()
        if any(record.get(key) != value for key, value in identity.items()):
            return None
        frame = pd.read_excel(path, sheet_name="runs")
        if len(frame) != int(identity["expected_rows"]):
            return None
        if checkpoint_content_sha256(frame) != record["content_sha256"]:
            return None
        observed = set(
            frame[["sample_size", "replicate", "method", "order_id"]]
            .itertuples(index=False, name=None)
        )
        expected = {
            (sample_size, replicate, method, order_id)
            for sample_size in settings["sample_sizes"]
            for replicate in range(1, int(settings["repetitions"]) + 1)
            for method, order_id in finite_sample_methods()
        }
        return frame if observed == expected else None
    except (OSError, ValueError, KeyError, TypeError):
        return None

# Compute a reproducible bootstrap 95% confidence interval for parameter-set-level statistics.
# Compute reproducible bootstrap interval estimates for parameter-set means.
def bootstrap_mean_ci(
    values: np.ndarray,
    seed: int,
    repetitions: int,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(repetitions, len(values)))
    means = values[indices].mean(axis=1)
    lower, upper = np.quantile(means, [0.025, 0.975])
    return float(lower), float(upper)

# Coordinate all parameter sets, reuse valid checkpoints, and generate two-level Bias/RMSE summaries.
# Coordinate all parameter sets and summarize Bias, RMSE, and confidence intervals.
def run_experiment2(
    project_root: Path,
    run_mode: str,
    experiment_code_sha256: str,
    resume: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    project_root = Path(project_root)
    settings = get_settings(run_mode)
    bank = ensure_shared_parameter_bank(project_root / "data" / "alarm_parameter_sets.xlsx")
    selected_ids = tuple(int(value) for value in settings["parameter_ids"])

    structure_hashes = {
        "J_G": edge_sha256(ALARM_DAG),
        **{
            f"FGCR_{order_id}": edge_sha256(result.H_R)
            for order_id, result in OBSERVATIONAL_STRUCTURES.items()
        },
        "IC_G_R": edge_sha256(G_R),
    }
    experiment_payload = {
        "sampling_J": "D_J~P(G)",
        "sampling_FGCR_IC": "one shared D_C~P(R|C) sampled from exact order_1 H_R",
        "bdeu_iss": ISS,
        "sample_sizes": settings["sample_sizes"],
        "repetitions": settings["repetitions"],
        "methods": finite_sample_methods(),
    }
    protocol_sha256 = experiment_protocol_sha256(
        2,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
    )

    frames: list[pd.DataFrame] = []
    checkpoint_paths: list[Path] = []
    for parameter_id in selected_ids:
        source_cpts = bank.cpts(parameter_id)
        parameter_hash = cpts_sha256(source_cpts, PARENT_ORDER)
        identity = checkpoint_identity(
            parameter_id,
            settings,
            parameter_hash,
            bank.content_sha256,
            protocol_sha256,
        )
        checkpoint = (
            project_root / "checkpoints"
            / f"02_{run_mode}_parameter_{parameter_id:03d}.xlsx"
        )
        checkpoint_paths.append(checkpoint)
        frame = read_valid_checkpoint(checkpoint, identity, settings) if resume else None
        if frame is None:
            frame = run_experiment2_parameter(
                parameter_id=parameter_id,
                source_cpts=source_cpts,
                settings=settings,
                parameter_bank_sha256=bank.content_sha256,
                protocol_sha256=protocol_sha256,
                run_mode=run_mode,
            )
            write_checkpoint(checkpoint, frame, identity)
        frames.append(frame)

    runs = pd.concat(frames, ignore_index=True)
    expected_rows = (
        len(selected_ids)
        * len(settings["sample_sizes"])
        * int(settings["repetitions"])
        * 6
    )
    if len(runs) != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows, observed {len(runs)}")

    # At the first level, compute Bias and RMSE for each parameter set across repetitions.
    per_parameter = (
        runs.groupby(
            ["parameter_id", "sample_size", "method", "order_id"],
            sort=False,
        )
        .agg(
            n_replicates=("replicate", "size"),
            bias=("error", "mean"),
            rmse=("squared_error", lambda values: float(np.sqrt(np.mean(values)))),
        )
        .reset_index()
    )

    # At the second level, aggregate across parameter sets and provide a parameter-set bootstrap CI for mean RMSE.
    summary_rows: list[dict[str, object]] = []
    bootstrap_repetitions = 500 if run_mode == "quick" else 2000
    for (sample_size, method, order_id), group in per_parameter.groupby(
        ["sample_size", "method", "order_id"], sort=False
    ):
        label = f"{sample_size}|{method}|{order_id}|{run_mode}"
        seed = int(hashlib.sha256(label.encode()).hexdigest()[:8], 16)
        lower, upper = bootstrap_mean_ci(
            group["rmse"].to_numpy(float), seed, bootstrap_repetitions
        )
        summary_rows.append({
            "sample_size": int(sample_size),
            "method": method,
            "order_id": order_id,
            "n_parameter_sets": group["parameter_id"].nunique(),
            "mean_bias": group["bias"].mean(),
            "median_bias": group["bias"].median(),
            "mean_rmse": group["rmse"].mean(),
            "median_rmse": group["rmse"].median(),
            "rmse_ci_lower_95": lower,
            "rmse_ci_upper_95": upper,
            "bootstrap_repetitions": bootstrap_repetitions,
            "bootstrap_seed": seed,
        })
    summary = pd.DataFrame(summary_rows)

    # Keep checkpoints by default; delete exactly the checkpoint files created for this run only when FGCR_KEEP_CHECKPOINTS=0 is explicitly set.
    if not KEEP_CHECKPOINTS:
        for checkpoint in checkpoint_paths:
            checkpoint.unlink(missing_ok=True)

    protocol = experiment_protocol_frame(
        2,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
        protocol_sha256,
    )
    return runs, per_parameter, summary, protocol, protocol_sha256


## 3. Output and Plotting


In [3]:
# Atomically save repetition-level records using deterministic gzip parameters to avoid storing the 420,000 formal-run rows in formatted Excel files.
def write_runs_csv_gzip_atomic(path: Path, runs: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    try:
        # Set mtime=0 to remove the timestamp from the gzip header, so identical runs produce reproducible file hashes.
        runs.to_csv(
            temporary,
            index=False,
            encoding="utf-8",
            compression={"method": "gzip", "compresslevel": 9, "mtime": 0},
        )
        os.replace(temporary, path)
    finally:
        # After a successful replacement, temporary no longer exists; on failure, remove the incomplete file without touching the previous formal result.
        if temporary.exists():
            temporary.unlink()

# Store repetition-level results in compressed form;
# Excel retains only summaries, structures, protocol, environment, and the runs-file manifest.
def save_experiment2(
    project_root: Path,
    runs: pd.DataFrame,
    per_parameter: pd.DataFrame,
    summary: pd.DataFrame,
    protocol: pd.DataFrame,
    protocol_sha256: str,
) -> tuple[Path, Path]:
    result_dir = Path(project_root) / "result"
    output = result_dir / "02_finite_sample_inference.xlsx"

    # A single output may contain only one run_mode; the file name therefore distinguishes quick, pilot, and formal runs.
    run_modes = tuple(runs["run_mode"].drop_duplicates())
    if len(run_modes) != 1:
        raise ValueError("Experiment 2 runs must contain exactly one run_mode")
    run_mode = str(run_modes[0])
    runs_output = result_dir / f"02_finite_sample_runs_{run_mode}.csv.gz"
    write_runs_csv_gzip_atomic(runs_output, runs)

    #  The manifest writes the row/column dimensions and byte-level hash of the external detail file back into Excel for integrity auditing.
    runs_manifest = pd.DataFrame([{
        "storage_format": "csv.gz",
        "relative_path": runs_output.name,
        "run_mode": run_mode,
        "row_count": len(runs),
        "column_count": len(runs.columns),
        "columns_json": json.dumps(list(runs.columns), ensure_ascii=False),
        "file_sha256": file_sha256(runs_output),
    }])
    write_excel_atomic(output, {
        "runs_manifest": runs_manifest,
        "per_parameter": per_parameter,
        "summary": summary,
        "structures": OBSERVATIONAL_STRUCTURE_AUDIT,
        "protocol": protocol,
        "environment": pd.DataFrame([environment_record()]),
    })
    verify_output_protocol(output, protocol_sha256)
    return output, runs_output

# Generate a four-panel figure showing the Bias/RMSE distributions at the largest sample size and the error trajectories across all sample sizes.
def plot_experiment2(
    project_root: Path,
    per_parameter: pd.DataFrame,
    summary: pd.DataFrame,
) -> Path:
    os.environ.setdefault("MPLCONFIGDIR", str(Path(project_root) / "result" / "matplotlib_cache"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    output = Path(project_root) / "result" / "02_finite_sample_bias_rmse.png"
    # Retain all four FGCR topological orders in the left-side boxplots to visually demonstrate that their distributions overlap almost completely.
    labels = ["J", "C_1", "C_2", "C_3", "C_4", "IC"]

    # Convert the method/order fields into unified plotting labels.
    def method_label(method: str, order_id: str) -> str:
        if method == "FGCR-C":
            return f"C_{order_id.split('_')[-1]}"
        return method

    plot_parameter = per_parameter.copy()
    plot_parameter["label"] = [
        method_label(method, order_id)
        for method, order_id in zip(plot_parameter["method"], plot_parameter["order_id"])
    ]
    plot_summary = summary.copy()
    plot_summary["label"] = [
        method_label(method, order_id)
        for method, order_id in zip(plot_summary["method"], plot_summary["order_id"])
    ]
    # Remove the dominant n^{-1/2} convergence trend in RMSE to compare the finite-sample error constants across methods.
    plot_summary["sqrt_n_scaled_mean_rmse"] = (
        np.sqrt(plot_summary["sample_size"].astype(float))
        * plot_summary["mean_rmse"].astype(float)
    )

    # Mean absolute bias must first take the absolute value of the signed Bias for each parameter set and then average across parameter sets.
    absolute_bias_summary = (
        plot_parameter.assign(absolute_bias=plot_parameter["bias"].abs())
        .groupby(["sample_size", "method", "order_id"], sort=False, as_index=False)
        .agg(mean_absolute_bias=("absolute_bias", "mean"))
    )
    largest_n = int(plot_parameter["sample_size"].max())

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for i, ax in enumerate(axes.flat):
        ax.text(0.5, -0.22, f'({chr(97+i)})', 
                transform=ax.transAxes,
                ha='center', va='top',
                fontsize=12)
    
    bias_data = [
        plot_parameter.loc[
            (plot_parameter["sample_size"] == largest_n)
            & (plot_parameter["label"] == label),
            "bias",
        ].to_numpy(float)
        for label in labels
    ]
    rmse_data = [
        plot_parameter.loc[
            (plot_parameter["sample_size"] == largest_n)
            & (plot_parameter["label"] == label),
            "rmse",
        ].to_numpy(float)
        for label in labels
    ]
    axes[0, 0].boxplot(bias_data, tick_labels=labels, showfliers=False)
    axes[1, 0].boxplot(rmse_data, tick_labels=labels, showfliers=False)
    axes[0, 0].axhline(0.0, linewidth=0.8)
    axes[0, 0].set_title(f"Bias across parameter sets at n={largest_n}")
    axes[1, 0].set_title(f"RMSE across parameter sets at n={largest_n}")
    axes[0, 0].tick_params(axis="x", rotation=25)
    axes[1, 0].tick_params(axis="x", rotation=25)
    
    line_specs = (
    ("J", "J", "none", "#1f77b4"),
    ("C-Method_1", "FGCR-C", "order_1", "#ff7f0e"),
    ("C-Method_2", "FGCR-C", "order_2", "#2ca02c"),
    ("C-Method_3", "FGCR-C", "order_3", "#d62728"),
    ("C-Method_4", "FGCR-C", "order_4", "#9467bd"),
    ("IC", "IC", "none", "#8c564b"),)
    for display_label, method, order_id, color in line_specs:
        #  The upper-right panel shows mean absolute bias to avoid cancellation between positive and negative Bias across parameter sets.
        bias_group = absolute_bias_summary.loc[
            (absolute_bias_summary["method"] == method)
            & (absolute_bias_summary["order_id"] == order_id)
        ].sort_values("sample_size")
        axes[0, 1].plot(
            bias_group["sample_size"],
            bias_group["mean_absolute_bias"],
            marker="o",
            color=color,
            label=display_label,
        )

        rmse_group = plot_summary.loc[
            (plot_summary["method"] == method)
            & (plot_summary["order_id"] == order_id)
        ].sort_values("sample_size")

        sample_sizes = rmse_group[
            "sample_size"
        ].to_numpy(float)

        scaled_mean_rmse = rmse_group[
            "sqrt_n_scaled_mean_rmse"
        ].to_numpy(float)

        axes[1, 1].plot(
            sample_sizes,
            scaled_mean_rmse,
            marker="o",
            color=color,
            label=display_label,
        )

    axes[0, 1].set_title("Mean absolute bias versus sample size")
    axes[0, 1].set_ylabel("Mean absolute bias")
    axes[1, 1].set_title(r"$\sqrt{n}$-scaled mean RMSE versus sample size")
    axes[1, 1].set_ylabel( r"$\sqrt{n}\times$ Mean RMSE")
    for axis in (axes[0, 1], axes[1, 1]):
        axis.set_xscale("log")
        axis.set_xlabel("Sample size")
    axes[0, 1].legend(frameon=False, fontsize=8)
    axes[1, 1].legend(frameon=False, fontsize=8)
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output


## 4. Execution


In [4]:
EXPERIMENT2_CODE_SHA256 = tagged_code_sha256(
    PROJECT_ROOT / "02_Finite_Sample_Inference.ipynb", "experiment2_core"
)
(
    experiment2_runs,
    experiment2_per_parameter,
    experiment2_summary,
    experiment2_protocol,
    experiment2_protocol_sha256,
) = run_experiment2(
    PROJECT_ROOT,
    RUN_MODE,
    EXPERIMENT2_CODE_SHA256,
    resume=RESUME_CHECKPOINTS,
)
experiment2_workbook, experiment2_runs_file = save_experiment2(
    PROJECT_ROOT,
    experiment2_runs,
    experiment2_per_parameter,
    experiment2_summary,
    experiment2_protocol,
    experiment2_protocol_sha256,
)
experiment2_figure = plot_experiment2(
    PROJECT_ROOT, experiment2_per_parameter, experiment2_summary
)
print(experiment2_summary.to_string(index=False))
print(f"Workbook: {experiment2_workbook}")
print(f"Full runs: {experiment2_runs_file}")
print(f"Figure:   {experiment2_figure}")


 sample_size method order_id  n_parameter_sets     mean_bias  median_bias  mean_rmse  median_rmse  rmse_ci_lower_95  rmse_ci_upper_95  bootstrap_repetitions  bootstrap_seed
         100      J     none               100  7.735598e-04     0.003186   0.016650     0.011371          0.013379          0.019917                   2000      4191678567
         100 FGCR-C  order_1               100  6.483018e-04     0.003065   0.012872     0.009373          0.010671          0.015399                   2000      3022719561
         100 FGCR-C  order_2               100  4.638848e-04     0.003050   0.013198     0.009727          0.011016          0.015647                   2000      2956876860
         100 FGCR-C  order_3               100  2.642696e-04     0.003177   0.014371     0.011062          0.012309          0.016653                   2000      1093247173
         100 FGCR-C  order_4               100  3.987178e-04     0.003512   0.015657     0.011894          0.013478          0.018034  

In [5]:
# Four panels:
# (a) Boxplots of Bias across sample sizes
# (b) Trend of mean Bias across sample sizes
# (c) Boxplots of RMSE across sample sizes
# (d) Trend of mean RMSE across sample sizes
#
# Statistical level:
# Each boxplot element is based on parameter-level Bias / RMSE,
# i.e., Bias / RMSE is first computed across replicates within each
# parameter × sample_size × method combination, and the resulting
# distribution is then displayed across different parameter sets.
from pathlib import Path
import os
import numpy as np
import pandas as pd

# 0. Project paths
PROJECT_ROOT = Path.cwd().resolve()
RESULT_DIR = PROJECT_ROOT / "result"
WORKBOOK_PATH = RESULT_DIR / "02_finite_sample_inference.xlsx"
FIGURE_PATH = RESULT_DIR / "02_finite_sample_bias_rmse_xie_style.png"

# 1. Check existing experiment results
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError( "未找到 Experiment 2 已有结果文件：\n"  f"{WORKBOOK_PATH}\n""请确认当前工作目录是项目根目录。" )
# Read only the two sheets required for plotting.
per_parameter = pd.read_excel(WORKBOOK_PATH,sheet_name="per_parameter",)
summary = pd.read_excel( WORKBOOK_PATH, sheet_name="summary",)
print("Loaded existing Experiment 2 results.")
print(f"Workbook       : {WORKBOOK_PATH}")
print(f"per_parameter  : {per_parameter.shape}")
print(f"summary        : {summary.shape}")

# 2. matplotlib
os.environ.setdefault( "MPLCONFIGDIR",str(RESULT_DIR / "matplotlib_cache"),)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 3. Method order
METHOD_ORDER = [ "C-Method_1","C-Method_2", "C-Method_3", "C-Method_4", "IC-Method", "J-Method",]

# 4. Convert internal method/order identifiers to plotting labels
def method_label(method: str, order_id: str) -> str:
    method = str(method)
    order_id = str(order_id)
    if method == "FGCR-C":
        order_number = order_id.split("_")[-1]
        return f"C-Method_{order_number}"
    if method == "IC": return "IC-Method"
    if method == "J": return "J-Method"
    raise ValueError( f"未知 method/order：method={method}, order_id={order_id}" )

# 5. Prepare existing results
required_per_parameter_columns = { "parameter_id", "sample_size", "method", "order_id", "bias", "rmse",}
required_summary_columns = { "sample_size", "method", "order_id", "mean_bias", "mean_rmse",}
missing_per_parameter = ( required_per_parameter_columns- set(per_parameter.columns))
missing_summary = (required_summary_columns- set(summary.columns))
if missing_per_parameter:raise KeyError("per_parameter 缺少必要列："f"{sorted(missing_per_parameter)}")
if missing_summary: raise KeyError("summary 缺少必要列："f"{sorted(missing_summary)}")

plot_parameter = per_parameter.copy()
plot_parameter["sample_size"] = (plot_parameter["sample_size"].astype(int))
plot_parameter["label"] = [
    method_label(method, order_id)
    for method, order_id in zip( plot_parameter["method"], plot_parameter["order_id"],)]
plot_summary = summary.copy()
plot_summary["sample_size"] = (plot_summary["sample_size"].astype(int))

plot_summary["label"] = [
    method_label(method, order_id)
    for method, order_id in zip( plot_summary["method"],plot_summary["order_id"],)]

absolute_bias_summary = ( plot_parameter.assign(absolute_bias=plot_parameter["bias"].abs())
    .groupby( ["sample_size", "method", "order_id", "label"], sort=False, as_index=False,)
    .agg(mean_absolute_bias=("absolute_bias", "mean")))
sample_sizes = sorted(plot_parameter["sample_size"].drop_duplicates().tolist())

# 6.  Data-integrity checks
observed_methods = set(plot_parameter["label"].drop_duplicates())

expected_methods = set(METHOD_ORDER)
if observed_methods != expected_methods:
    raise AssertionError(  "实验方法与预期不一致。\n"  f"Expected: {sorted(expected_methods)}\n"f"Observed: {sorted(observed_methods)}" )

# Each n × method combination should contain the same number of parameter sets.
parameter_counts = (plot_parameter.groupby(["sample_size", "label"], sort=False,)["parameter_id"].nunique().unstack())

if parameter_counts.isna().any().any():
    raise AssertionError( "某些 sample_size × method 组合缺失。")
print("\nSample sizes:")
print(sample_sizes)
print("\nNumber of parameter sets per sample size/method:")
print(parameter_counts)

# 7. Grouped boxplots: two panels on the left
def draw_grouped_boxplot(
    ax,
    data: pd.DataFrame,
    value_column: str,
    sample_sizes: list[int],
    method_order: list[str],):
    centers = np.arange( len(sample_sizes), dtype=float,  )
    n_methods = len(method_order)
    # Positions of the six methods within each sample-size group.
    offsets = np.linspace(-0.30, 0.30, n_methods, )
    box_width = 0.085
    colors = ( plt.rcParams["axes.prop_cycle"] .by_key()["color"]  )
    legend_handles = []
    for method_index, method in enumerate( method_order):
        positions = []
        values_by_sample_size = []
        for sample_index, sample_size in enumerate( sample_sizes ):
            values = (data.loc[(data["sample_size"]== sample_size )  &  (data["label"] == method ),value_column, ]
                .dropna().to_numpy(dtype=float) )

            if len(values) == 0:
                raise AssertionError( f"没有找到数据：" f"{value_column}, "f"n={sample_size}, "f"method={method}")

            values_by_sample_size.append( values )
            positions.append(centers[sample_index] + offsets[method_index])
        color = colors[method_index % len(colors)]

        box_result = ax.boxplot(
            values_by_sample_size,
            positions=positions,
            widths=box_width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=True,
            flierprops={"marker": "o", "markersize": 3.0,"markerfacecolor": "none",
                "markeredgecolor": "0.35","markeredgewidth": 0.55, "linestyle": "none", },
            medianprops={"linewidth": 0.9,},
            whiskerprops={"linewidth": 0.7,},
            capprops={"linewidth": 0.7, },
            boxprops={ "linewidth": 0.7, }, )
        
        for box in box_result["boxes"]:
            box.set_facecolor(color)
            box.set_alpha(0.75)
        legend_handles.append(  Line2D([0], [0],marker="s",linestyle="none",markerfacecolor=color,
                markeredgecolor="0.3", markersize=6, label=method,)  )
        
    ax.set_xticks(  centers, [str(n) for n in sample_sizes], )
    ax.set_xlim(centers[0] - 0.55,centers[-1] + 0.55,  )
    return legend_handles

# 8.Trend plots: two panels on the right
def draw_mean_trend(ax,data: pd.DataFrame, y_column: str,method_order: list[str],):
    colors = (plt.rcParams["axes.prop_cycle"].by_key()["color"] )
    markers = [ "o", "^","s","+", "x", "*", ]
    for method_index, method in enumerate(method_order ):
        group = (data.loc[data["label"] == method].sort_values("sample_size"))
        if group.empty:
            raise AssertionError(f"summary 中不存在方法：{method}")

        ax.plot(group["sample_size"].to_numpy(dtype=float),
            group[y_column].to_numpy(dtype=float),
            linestyle="--",
            linewidth=0.9,
            marker=markers[method_index],
            markersize=5,
            color=colors[ method_index % len(colors)],
            label=method, )



# 9.  Generate the four-panel figure
def plot_experiment2_xie_style(per_parameter: pd.DataFrame, summary: pd.DataFrame,output_path: Path,) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir( parents=True, exist_ok=True, )
    fig, axes = plt.subplots( 2, 2, figsize=(14.5, 8.6),)

    ax_bias_box = axes[0, 0]
    ax_bias_trend = axes[0, 1]
    ax_rmse_box = axes[1, 0]
    ax_rmse_trend = axes[1, 1]
    # (a) Bias boxplots
    legend_handles = draw_grouped_boxplot( ax=ax_bias_box, data=per_parameter,  
                                          value_column="bias",  sample_sizes=sample_sizes, method_order=METHOD_ORDER,)
    ax_bias_box.axhline( 0.0, linewidth=0.7, )
    ax_bias_box.set_xlabel(  "Sample Size" )
    ax_bias_box.set_ylabel( "Bias" )
    ax_bias_box.legend( handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, 
                       frameon=False,fontsize=8, columnspacing=0.8,handletextpad=0.3, )
    # (b) Mean Absolute Bias trend
    draw_mean_trend( ax=ax_bias_trend, data=absolute_bias_summary,y_column="mean_absolute_bias",method_order=METHOD_ORDER, )
    ax_bias_trend.set_xlabel("Sample Size" )
    ax_bias_trend.set_ylabel(  "Mean Absolute Bias" )
    ax_bias_trend.legend( loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3,frameon=False,fontsize=8,
                         columnspacing=0.8, handletextpad=0.3, )
    # (c) RMSE boxplots
    draw_grouped_boxplot( ax=ax_rmse_box, data=per_parameter, value_column="rmse", sample_sizes=sample_sizes, method_order=METHOD_ORDER, )
    ax_rmse_box.set_xlabel( "Sample Size" )
    ax_rmse_box.set_ylabel( "RMSE" )
    ax_rmse_box.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        frameon=False,
        fontsize=8,
        columnspacing=0.8,
        handletextpad=0.3,)
    # (d) Mean RMSE trend
    draw_mean_trend(  ax=ax_rmse_trend,data=summary,
        y_column="mean_rmse", method_order=METHOD_ORDER, )
    ax_rmse_trend.set_xlabel(   "Sample Size" )
    ax_rmse_trend.set_ylabel("RMSE" )
#     ax_rmse_trend.set_ylim(0.00, 0.006)
#     ax_rmse_trend.set_yticks([0.002, 0.004, 0.006])
    ax_rmse_trend.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        frameon=False,
        fontsize=8,
        columnspacing=0.8,
        handletextpad=0.3, )

    # 10. Use the actual n values as ticks on the right-side panels
    max_sample_size = max(sample_sizes)
    preferred_ticks = [ 0, 2500,  5000, 7500, 10000,]
    trend_ticks = [
        tick
        for tick in preferred_ticks
        if tick <= max_sample_size  ]

    if max_sample_size not in trend_ticks:
        trend_ticks.append( max_sample_size)

    trend_ticks = sorted(  set(trend_ticks)  )
    ax_bias_trend.set_xticks( trend_ticks  )
    ax_rmse_trend.set_xticks(   trend_ticks )
    # 11. Grid settings
    for ax in axes.flat:
        ax.grid( True,linewidth=0.4,alpha=0.20,)
        ax.tick_params( axis="both",labelsize=9,)
        ax.set_title("")
    # 12. Four panel caption
    panel_captions = [
        "(a) The boxplots of the biases",
        "(b)The trend of mean absolute biases",
        "(c) The boxplots of the RMSEs",
        "(d) The trend of average value of the RMSEs",  ]

    for ax, caption in zip(
        [ ax_bias_box,ax_bias_trend,ax_rmse_box,ax_rmse_trend,],
        panel_captions,):
        ax.text( 0.5, -0.20, caption,transform=ax.transAxes,
            ha="center", va="top", fontsize=12, family="serif",)
    # 13. Spacing between panels
    fig.subplots_adjust(  left=0.07,  right=0.985,  bottom=0.10,  top=0.93,  wspace=0.15,     hspace=0.50,)
    # 14. Save the figure
    fig.savefig( output_path, dpi=600, bbox_inches="tight", facecolor="white", )
    plt.close(fig)
    return output_path
figure_path = plot_experiment2_xie_style(
    per_parameter=plot_parameter,
    summary=plot_summary,
    output_path=FIGURE_PATH,)
print("\nExperiment 2 figure regenerated successfully.")
print(f"Figure: {figure_path}")

Loaded existing Experiment 2 results.
Workbook       : D:\jupyternoteboook\实验\条件概率实验\result\02_finite_sample_inference.xlsx
per_parameter  : (4200, 7)
summary        : (42, 12)

Sample sizes:
[100, 500, 1000, 2500, 5000, 7500, 10000]

Number of parameter sets per sample size/method:
label        J-Method  C-Method_1  C-Method_2  C-Method_3  C-Method_4  \
sample_size                                                             
100               100         100         100         100         100   
500               100         100         100         100         100   
1000              100         100         100         100         100   
2500              100         100         100         100         100   
5000              100         100         100         100         100   
7500              100         100         100         100         100   
10000             100         100         100         100         100   

label        IC-Method  
sample_size             
100     